In [3]:
import sys
import pandas as pd
import numpy as np
import random
import tensorflow as tf
import plotly.express as px
import plotly.graph_objects as go
import os
os.environ["PYTHONHASHSEED"] = "42"
os.environ["TF_DETERMINISTIC_OPS"] = "1"
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from pathlib import Path

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "src").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.simulation.dgp0 import Tier0Config, simulate_panel
from src.simulation.validation import plot_market_plotly, separation_auc_like
from src.data.feature_eng import feature_eng_syn
from src.simulation.windows.windows import make_windows
from src.data.cartel_data import prepare_cartel_panel

In [4]:
df1 = pd.read_csv("../data/cartel_data/cartel_prices_monthly_combined.csv")
df1.shape

(191, 5)

In [5]:
df2 = pd.read_csv("../data/cartel_data/vitamin_data.csv")
df2.shape

(576, 5)

In [ ]:
cartel_1 = prepare_cartel_panel(df1)
cartel_1.head()

,market_id,date,t,p,S,collusion_flag,unit,price,raw_price
0,citric_acid,1987-01-01,0,4.369448,0,0,cents_per_lb,79.00,79.00
1,citric_acid,1987-02-01,1,4.366278,0,0,cents_per_lb,78.75,78.75
2,citric_acid,1987-03-01,2,4.363353,0,0,cents_per_lb,78.52,78.52
3,citric_acid,1987-04-01,3,4.360037,0,0,cents_per_lb,78.26,78.26
4,citric_acid,1987-05-01,4,4.356837,0,0,cents_per_lb,78.01,78.01


In [16]:
cartel_2 = prepare_cartel_panel(df2)
cartel_2.head()

,market_id,date,t,p,S,collusion_flag,unit,price_index,raw_price
0,beta_carotene,1990-01-01,0,4.248495,2,1,index_Jan1995_100,70.000000,70.000000
1,beta_carotene,1990-02-01,1,4.254543,2,1,index_Jan1995_100,70.424658,70.424658
2,beta_carotene,1990-03-01,2,4.259975,2,1,index_Jan1995_100,70.808219,70.808219
3,beta_carotene,1990-04-01,3,4.265954,2,1,index_Jan1995_100,71.232877,71.232877
4,beta_carotene,1990-05-01,4,4.271707,2,1,index_Jan1995_100,71.643836,71.643836


In [18]:
cartel_df = pd.concat([cartel_1, cartel_2])
cartel_df.shape

(767, 10)

In [19]:
cartel_windows = make_windows(
    cartel_df,
    window=18,
    price_col="p",
    state_col="S",
    id_col="market_id",
    time_col="t",
)

In [20]:
cartel_windows.head()

,market_id,window_start,window_end,window_length,Price 1,Price 2,Price 3,Price 4,Price 5,Price 6,...,Price 14,Price 15,Price 16,Price 17,Price 18,share_C,share_T,share_K,state_mode,is_pure_80
0,beta_carotene,0,17,18,4.248495,4.254543,4.259975,4.265954,4.271707,4.277617,...,4.325384,4.332462,4.340241,4.347712,4.355373,0.0,0.0,1.0,2,1.0
1,beta_carotene,1,18,18,4.254543,4.259975,4.265954,4.271707,4.277617,4.283303,...,4.332462,4.340241,4.347712,4.355373,4.362732,0.0,0.0,1.0,2,1.0
2,beta_carotene,2,19,18,4.259975,4.265954,4.271707,4.277617,4.283303,4.289145,...,4.340241,4.347712,4.355373,4.362732,4.370280,0.0,0.0,1.0,2,1.0
3,beta_carotene,3,20,18,4.265954,4.271707,4.277617,4.283303,4.289145,4.294953,...,4.347712,4.355373,4.362732,4.370280,4.377771,0.0,0.0,1.0,2,1.0
4,beta_carotene,4,21,18,4.271707,4.277617,4.283303,4.289145,4.294953,4.300542,...,4.355373,4.362732,4.370280,4.377771,4.384968,0.0,0.0,1.0,2,1.0


In [21]:
cartel_windows["cartel_share"] = cartel_windows["share_K"]
cartel_windows["cartel_window"] = (cartel_windows["share_K"] >= 0.5).astype(int)
cartel_windows["pure_cartel_window"] = (cartel_windows["share_K"] >= 0.8).astype(int)

In [22]:
cartel_windows.head()

,market_id,window_start,window_end,window_length,Price 1,Price 2,Price 3,Price 4,Price 5,Price 6,...,Price 17,Price 18,share_C,share_T,share_K,state_mode,is_pure_80,cartel_share,cartel_window,pure_cartel_window
0,beta_carotene,0,17,18,4.248495,4.254543,4.259975,4.265954,4.271707,4.277617,...,4.347712,4.355373,0.0,0.0,1.0,2,1.0,1.0,1,1
1,beta_carotene,1,18,18,4.254543,4.259975,4.265954,4.271707,4.277617,4.283303,...,4.355373,4.362732,0.0,0.0,1.0,2,1.0,1.0,1,1
2,beta_carotene,2,19,18,4.259975,4.265954,4.271707,4.277617,4.283303,4.289145,...,4.362732,4.370280,0.0,0.0,1.0,2,1.0,1.0,1,1
3,beta_carotene,3,20,18,4.265954,4.271707,4.277617,4.283303,4.289145,4.294953,...,4.370280,4.377771,0.0,0.0,1.0,2,1.0,1.0,1,1
4,beta_carotene,4,21,18,4.271707,4.277617,4.283303,4.289145,4.294953,4.300542,...,4.377771,4.384968,0.0,0.0,1.0,2,1.0,1.0,1,1


In [23]:
cartel_windows["cartel_window"].value_counts()


cartel_window
1    482
0    183
Name: count, dtype: int64